# Enum — Nível Intermediário

**Objetivo:** escolher o tipo certo da família `enum`, dar comportamento aos membros, aceitar entradas tolerantes e
integrar Enums com configuração, JSON, dataclasses e Pydantic.

Roteiro:

1. `StrEnum` e `IntEnum`: quando o membro precisa "ser" o valor
2. `str()`, `format()` e f-strings: a diferença que quebra logs e SQL
3. `Flag`: combinando opções (e a pegadinha do alias de combinação)
4. `IntFlag`
5. Métodos, propriedades e `classmethod` no Enum
6. Membros ricos: valor + atributos com `__init__`
7. `_missing_`: entrada tolerante (case-insensitive, sinônimos)
8. `_generate_next_value_` e `@verify`
9. Enum em **configuração** (env vars, YAML)
10. Enum em **JSON**, **dataclass** e **Pydantic** (DTOs de API + JSON Schema)

In [1]:
import json, os
from enum import Enum, IntEnum, StrEnum, Flag, IntFlag, auto, unique, verify, UNIQUE, CONTINUOUS, NAMED_FLAGS

## 1. `StrEnum` e `IntEnum`

`StrEnum` (3.11+) é **subclasse de `str`**: o membro é uma string de verdade. `IntEnum` é subclasse de `int`.
Isso permite comparar direto com o valor bruto e passar o membro para qualquer função que espera `str`/`int`.

In [2]:
class Environment(StrEnum):
    DEV = auto()          # em StrEnum, auto() gera o nome em minúsculas
    STAGING = auto()
    PROD = auto()

class HttpStatus(IntEnum):
    OK = 200
    CREATED = 201
    BAD_REQUEST = 400
    NOT_FOUND = 404

print(list(Environment))
print(Environment.PROD == "prod", isinstance(Environment.PROD, str), Environment.PROD.upper())
print(HttpStatus.NOT_FOUND == 404, HttpStatus.NOT_FOUND + 1, HttpStatus(201).name)
print(sorted([HttpStatus.NOT_FOUND, HttpStatus.OK, HttpStatus.BAD_REQUEST]))

[<Environment.DEV: 'dev'>, <Environment.STAGING: 'staging'>, <Environment.PROD: 'prod'>]
True True PROD
True 405 CREATED
[<HttpStatus.OK: 200>, <HttpStatus.BAD_REQUEST: 400>, <HttpStatus.NOT_FOUND: 404>]


O preço: você perde parte da segurança. Com `IntEnum`, `HttpStatus.OK == 200 == Priority.X` (se algum outro IntEnum valer 200)
— enums diferentes passam a ser "iguais" via o inteiro. Use `IntEnum` só quando precisar interoperar com APIs numéricas.

In [3]:
class Weekday(IntEnum):
    MONDAY = 1

class Level(IntEnum):
    LOW = 1

print("enums diferentes iguais via int:", Weekday.MONDAY == Level.LOW)

enums diferentes iguais via int: True


## 2. `str()`, `format()` e f-strings

Diferença sutil que já quebrou muita query SQL e chave de cache:

In [4]:
class PlainColor(Enum):
    RED = "red"

class StrColor(StrEnum):
    RED = "red"

for m in (PlainColor.RED, StrColor.RED, HttpStatus.OK):
    print(f"{type(m).__name__:<11} str()={str(m)!r:<18} f-string={f'{m}'!r:<18} .value={m.value!r}")

PlainColor  str()='PlainColor.RED'   f-string='PlainColor.RED'   .value='red'
StrColor    str()='red'              f-string='red'              .value='red'
HttpStatus  str()='200'              f-string='200'              .value=200


Regra prática: quando o destino é texto externo (SQL, URL, chave de cache, JSON), use **`.value` explicitamente** —
funciona igual para todos os tipos e deixa a intenção clara.

## 3. `Flag`: combinando opções

`Enum` representa **um** estado por vez. `Flag` representa um **conjunto** de opções ativas, usando bits:
cada membro é uma potência de 2 e a combinação é feita com `|`.

| operação | significado |
|---|---|
| `a \| b` | união (ativa ambos) |
| `a & b` | interseção |
| `x in combo` | o bit `x` está ativo? |
| `~a` | complemento (todos os outros) |
| `combo ^ a` | alterna `a` |
| `Flag(0)` | nenhum ativo (falsy) |

In [5]:
class Permission(Flag):
    READ = auto()        # 1
    WRITE = auto()       # 2
    DELETE = auto()      # 4
    SHARE = auto()       # 8

    EDITOR = READ | WRITE                  # combinação nomeada (alias)
    ADMIN = READ | WRITE | DELETE | SHARE

teacher = Permission.READ | Permission.WRITE | Permission.DELETE
student = Permission.READ

print(teacher, "| valor:", teacher.value)
print("DELETE in teacher:", Permission.DELETE in teacher)
print("WRITE in student :", Permission.WRITE in student)
print("student tem tudo de EDITOR?", Permission.EDITOR in student)
print("teacher sem DELETE:", teacher & ~Permission.DELETE)
print("membros individuais de teacher:", list(teacher))
print("iterando a classe:", list(Permission), "  <- combinações nomeadas não aparecem")
print("nenhuma permissão:", Permission(0), bool(Permission(0)))

Permission.READ|WRITE|DELETE | valor: 7
DELETE in teacher: True
WRITE in student : False
student tem tudo de EDITOR? False
teacher sem DELETE: Permission.EDITOR
membros individuais de teacher: [<Permission.READ: 1>, <Permission.WRITE: 2>, <Permission.DELETE: 4>]
iterando a classe: [<Permission.READ: 1>, <Permission.WRITE: 2>, <Permission.DELETE: 4>, <Permission.SHARE: 8>]   <- combinações nomeadas não aparecem
nenhuma permissão: Permission(0) False


### A pegadinha do alias de combinação

Se um membro de `Flag` recebe um valor que **não é potência de 2**, ele não é uma opção nova: é um **apelido da combinação**
dos bits correspondentes. Veja o que acontece quando alguém define `BLACK = 12` achando que criou uma cor nova:

In [6]:
class Color(Flag):
    RED = 1
    GREEN = 2
    BLUE = 4
    YELLOW = 8
    BLACK = 12         # 12 = 8 + 4 -> é YELLOW | BLUE !

combo = Color.YELLOW | Color.BLUE
print("YELLOW | BLUE =", combo)                  # imprime Color.BLACK
print("BLACK é a combinação?", Color.BLACK is combo)
print("BLACK aparece na iteração?", list(Color))
print("bits de BLACK:", list(Color.BLACK))

YELLOW | BLUE = Color.BLACK
BLACK é a combinação? True
BLACK aparece na iteração? [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 4>, <Color.YELLOW: 8>]
bits de BLACK: [<Color.BLUE: 4>, <Color.YELLOW: 8>]


Para impedir isso, use `@verify(NAMED_FLAGS)` (garante que todo bit usado por uma combinação tem nome) e, para opções
independentes, sempre `auto()` em vez de números escritos à mão. Se `BLACK` for de fato uma opção independente, ela precisa de bit próprio (16).

## 4. `IntFlag`

Como `Flag`, mas também é `int`: interopera com bitmasks numéricas (permissões de arquivo, flags de APIs em C, colunas inteiras no banco).

In [7]:
class FileMode(IntFlag):
    X = 1
    W = 2
    R = 4

mode = FileMode.R | FileMode.W
print(mode, int(mode), oct(FileMode.R | FileMode.W | FileMode.X))
print("aceita int direto:", FileMode(5), "| combina com int:", mode | 1)

6 6 0o7
aceita int direto: 5 | combina com int: 7


## 5. Métodos, propriedades e `classmethod`

Um Enum é uma classe: pode ter métodos. Isso mantém **regras ligadas ao vocabulário** perto dele, em vez de espalhadas em `if`s.

In [8]:
class OrderStatus(StrEnum):
    PENDING = auto()
    PAID = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()

    @property
    def is_final(self) -> bool:
        return self in {OrderStatus.DELIVERED, OrderStatus.CANCELLED}

    @property
    def label_pt(self) -> str:
        return {"pending": "Pendente", "paid": "Pago", "shipped": "Enviado",
                "delivered": "Entregue", "cancelled": "Cancelado"}[self.value]

    def can_cancel(self) -> bool:
        return self in {OrderStatus.PENDING, OrderStatus.PAID}

    @classmethod
    def open_statuses(cls) -> list["OrderStatus"]:
        return [s for s in cls if not s.is_final]

print(OrderStatus.PAID.label_pt, OrderStatus.PAID.can_cancel(), OrderStatus.SHIPPED.can_cancel())
print("abertos:", OrderStatus.open_statuses())
print("opções para um <select>:", [(s.value, s.label_pt) for s in OrderStatus])

Pago True False
abertos: [<OrderStatus.PENDING: 'pending'>, <OrderStatus.PAID: 'paid'>, <OrderStatus.SHIPPED: 'shipped'>]
opções para um <select>: [('pending', 'Pendente'), ('paid', 'Pago'), ('shipped', 'Enviado'), ('delivered', 'Entregue'), ('cancelled', 'Cancelado')]


## 6. Membros ricos: valor + atributos

Quando cada membro precisa de vários dados (código, descrição, limite, custo...), use uma tupla como valor e um `__init__`
para dar nome aos pedaços. O `value` continua sendo a tupla, a menos que você defina `_value_` no `__new__`.

In [9]:
class Plan(Enum):
    #        (código,  preço mensal, max usuários, suporte)
    FREE = ("free", 0, 1, "comunidade")
    PRO = ("pro", 49, 5, "email")
    TEAM = ("team", 199, 50, "chat")

    def __init__(self, code: str, monthly_price: int, max_users: int, support: str):
        self.code = code
        self.monthly_price = monthly_price
        self.max_users = max_users
        self.support = support

    def fits(self, users: int) -> bool:
        return users <= self.max_users

print(Plan.PRO.code, Plan.PRO.monthly_price, Plan.PRO.support)
print("menor plano para 12 usuários:", next(p for p in Plan if p.fits(12)))
print("value é a tupla:", Plan.PRO.value)

pro 49 email
menor plano para 12 usuários: Plan.TEAM
value é a tupla: ('pro', 49, 5, 'email')


Para que o **valor** seja só o código (e `Plan("pro")` funcione), defina `__new__` e atribua `_value_`:

In [10]:
class Region(Enum):
    def __new__(cls, code: str, latency_ms: int, gdpr: bool):
        obj = object.__new__(cls)
        obj._value_ = code                  # o valor "oficial" do membro
        obj.latency_ms = latency_ms
        obj.gdpr = gdpr
        return obj

    SA_EAST = ("sa-east-1", 20, False)
    US_EAST = ("us-east-1", 120, False)
    EU_WEST = ("eu-west-1", 190, True)

print(Region("eu-west-1"), Region.EU_WEST.value, Region.EU_WEST.gdpr)
print("mais próxima:", min(Region, key=lambda r: r.latency_ms))

Region.EU_WEST eu-west-1 True
mais próxima: Region.SA_EAST


## 7. `_missing_`: entrada tolerante

`_missing_` é chamado quando `Enum(valor)` não encontra o valor. Devolva um membro para aceitar a entrada, ou `None` para manter o erro.
Ideal para **normalizar** variações vindas de humanos e sistemas legados — num lugar só.

In [11]:
class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"

    @classmethod
    def _missing_(cls, value):
        if isinstance(value, str):
            v = value.strip().lower()
            synonyms = {"warn": "warning", "err": "error", "information": "info"}
            v = synonyms.get(v, v)
            for member in cls:
                if member.value == v:
                    return member
        return None

for raw in ["INFO", " Warn ", "err", "debug"]:
    print(f"{raw!r:<10} -> {LogLevel(raw)!r}")
try:
    LogLevel("verbose")
except ValueError as e:
    print("ValueError:", e)

'INFO'     -> <LogLevel.INFO: 'info'>
' Warn '   -> <LogLevel.WARNING: 'warning'>
'err'      -> <LogLevel.ERROR: 'error'>
'debug'    -> <LogLevel.DEBUG: 'debug'>
ValueError: 'verbose' is not a valid LogLevel


## 8. `_generate_next_value_` e `@verify`

### Controlando o `auto()`
`_generate_next_value_` define o que `auto()` produz. Precisa ser declarado **antes** dos membros.

In [12]:
class Event(StrEnum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower().replace("__", ".")      # ORDER__PAID -> order.paid

    ORDER__CREATED = auto()
    ORDER__PAID = auto()
    CUSTOMER__CREATED = auto()

print(list(Event))

[<Event.ORDER__CREATED: 'order.created'>, <Event.ORDER__PAID: 'order.paid'>, <Event.CUSTOMER__CREATED: 'customer.created'>]


### Validando a definição com `@verify` (3.11+)

| verificação | garante |
|---|---|
| `UNIQUE` | nenhum alias (como `@unique`) |
| `CONTINUOUS` | valores inteiros sem buracos (útil para códigos sequenciais) |
| `NAMED_FLAGS` | todo bit usado em combinações de `Flag` tem nome próprio |

In [13]:
for name, build in {
    "CONTINUOUS": lambda: verify(CONTINUOUS)(Enum("Step", {"ONE": 1, "TWO": 2, "FOUR": 4})),
    "NAMED_FLAGS": lambda: verify(NAMED_FLAGS)(Flag("Opt", {"A": 1, "B": 2, "AB_AND_C": 7})),
    "UNIQUE": lambda: verify(UNIQUE)(Enum("Tier", {"GOLD": 1, "OURO": 1})),
}.items():
    try:
        build()
    except ValueError as e:
        print(f"{name:<12} ValueError: {e}")

CONTINUOUS   ValueError: invalid enum 'Step': missing values 3
NAMED_FLAGS  ValueError: invalid Flag 'Opt': alias AB_AND_C is missing value 0x4 [use enum.show_flag_values(value) for details]
UNIQUE       ValueError: aliases found in <enum 'Tier'>: OURO -> GOLD


## 9. Enum em configuração

Variáveis de ambiente e arquivos de config são texto. Converter para Enum **no carregamento** garante que o resto do código
só lida com opções válidas — e o erro aparece no startup, listando as opções.

In [14]:
class CacheBackend(StrEnum):
    MEMORY = auto()
    REDIS = auto()
    DISABLED = auto()

class Feature(Flag):
    SIGNUP = auto()
    EXPORT = auto()
    BETA_UI = auto()
    AI_ASSISTANT = auto()

    @classmethod
    def parse(cls, raw: str) -> "Feature":
        """'signup, export' -> Feature.SIGNUP | Feature.EXPORT"""
        result = cls(0)
        for name in filter(None, (p.strip().upper() for p in raw.split(","))):
            try:
                result |= cls[name]
            except KeyError:
                raise ValueError(f"feature {name!r} desconhecida. Opções: {[m.name.lower() for m in cls]}") from None
        return result

def enum_from_env(var: str, enum_cls: type[Enum], default: Enum) -> Enum:
    raw = os.environ.get(var)
    if raw is None:
        return default
    try:
        return enum_cls(raw)
    except ValueError:
        options = ", ".join(str(m.value) for m in enum_cls)
        raise SystemExit(f"config inválida: {var}={raw!r}. Opções: {options}") from None

os.environ.update({"APP_ENV": "PROD", "APP_LOG_LEVEL": "warn", "APP_CACHE": "redis", "APP_FEATURES": "signup, ai_assistant"})

class Env(StrEnum):
    DEV = auto(); STAGING = auto(); PROD = auto()
    @classmethod
    def _missing_(cls, value):
        return cls.__members__.get(str(value).upper())

config = {
    "env": enum_from_env("APP_ENV", Env, Env.DEV),
    "log_level": enum_from_env("APP_LOG_LEVEL", LogLevel, LogLevel.INFO),
    "cache": enum_from_env("APP_CACHE", CacheBackend, CacheBackend.MEMORY),
    "features": Feature.parse(os.environ["APP_FEATURES"]),
}
print(config)
print("assistente ligado?", Feature.AI_ASSISTANT in config["features"])

os.environ["APP_CACHE"] = "memcached"
try:
    enum_from_env("APP_CACHE", CacheBackend, CacheBackend.MEMORY)
except SystemExit as e:
    print(e)
for k in ("APP_ENV", "APP_LOG_LEVEL", "APP_CACHE", "APP_FEATURES"):
    os.environ.pop(k)

{'env': <Env.PROD: 'prod'>, 'log_level': <LogLevel.WARNING: 'warning'>, 'cache': <CacheBackend.REDIS: 'redis'>, 'features': <Feature.SIGNUP|AI_ASSISTANT: 9>}
assistente ligado? True
config inválida: APP_CACHE='memcached'. Opções: memory, redis, disabled


Com `pydantic-settings` (veja `pydantic/02`) isso é automático: basta anotar o campo com o Enum.

## 10. Enum em JSON, dataclass e Pydantic

### JSON puro
`json.dumps` serializa `StrEnum`/`IntEnum` (são `str`/`int`), mas **não** `Enum` puro nem `Flag`:

In [15]:
print(json.dumps({"env": Env.PROD, "status": HttpStatus.OK}))
try:
    json.dumps({"color": PlainColor.RED})
except TypeError as e:
    print("TypeError:", e)
print(json.dumps({"color": PlainColor.RED}, default=lambda o: o.value if isinstance(o, Enum) else str(o)))

{"env": "prod", "status": 200}
TypeError: Object of type PlainColor is not JSON serializable
{"color": "red"}


### Dataclass
Dataclass não converte: se o dado vem de fora, converta no `__post_init__` (ou na borda).

In [16]:
from dataclasses import dataclass, asdict

@dataclass
class Order:
    id: int
    status: OrderStatus

    def __post_init__(self):
        self.status = OrderStatus(self.status)       # aceita "paid" ou OrderStatus.PAID

o = Order(1, "paid")
print(o, "| is_final:", o.status.is_final)
print(asdict(o), "->", json.dumps(asdict(o)))

Order(id=1, status=<OrderStatus.PAID: 'paid'>) | is_final: False
{'id': 1, 'status': <OrderStatus.PAID: 'paid'>} -> {"id": 1, "status": "paid"}


### Pydantic: validação, mensagens e JSON Schema de graça
Anotar o campo com o Enum faz o Pydantic: converter o valor, rejeitar valores fora do conjunto listando as opções e
publicar `"enum": [...]` no JSON Schema (OpenAPI). `use_enum_values=True` guarda o `.value` em vez do membro.

In [17]:
from pydantic import BaseModel, ConfigDict, ValidationError

class OrderOut(BaseModel):
    id: int
    status: OrderStatus
    last_http_status: HttpStatus | None = None

class OrderFlat(BaseModel):
    model_config = ConfigDict(use_enum_values=True)
    id: int
    status: OrderStatus

m = OrderOut.model_validate_json('{"id": 1, "status": "shipped"}')
print(repr(m.status), "| JSON:", m.model_dump_json())
print("use_enum_values:", repr(OrderFlat(id=1, status="paid").status))

try:
    OrderOut(id=1, status="lost")
except ValidationError as e:
    print(e.errors(include_url=False)[0]["msg"])

print(json.dumps(OrderOut.model_json_schema()["$defs"]["OrderStatus"]))

<OrderStatus.SHIPPED: 'shipped'> | JSON: {"id":1,"status":"shipped","last_http_status":null}
use_enum_values: 'paid'
Input should be 'pending', 'paid', 'shipped', 'delivered' or 'cancelled'


{"enum": ["pending", "paid", "shipped", "delivered", "cancelled"], "title": "OrderStatus", "type": "string"}


## Resumo

| precisa de... | use |
|---|---|
| vocabulário seguro, sem confundir com valor bruto | `Enum` |
| membro que **é** string (JSON, headers, chaves) | `StrEnum` |
| interoperar com códigos numéricos | `IntEnum` |
| várias opções ao mesmo tempo | `Flag` (com `auto()` e `@verify(NAMED_FLAGS)`) |
| bitmask numérica | `IntFlag` |
| regras ligadas ao vocabulário | métodos / `@property` no Enum |
| vários dados por membro | `__init__` (ou `__new__` + `_value_`) |
| entrada tolerante | `_missing_` |
| texto externo (SQL, cache, URL) | sempre `.value` |
| validação + schema | campo Pydantic anotado com o Enum |

### Exercícios
1. Crie `Currency(StrEnum)` com membros ricos (`symbol`, `decimals`) e um método `format(amount)`.
2. Faça `Permission.parse("read,write")` aceitar também `"*"` = todas.
3. Escreva `_missing_` para `HttpStatus` que aceite `"404"` (string) além de `404`.

➡️ Próximo: [`03_enum_avancado.ipynb`](03_enum_avancado.ipynb)